## Introduction
In this Colab Notebook, we are going to explore Mistral-7B-Instruct-v0.3, a model fine-tuned for generating text & chatting.

By the end of this tutorial, you'll be able to interact with this model and use it to generate conversational responses.

Whether you're curious about chatbot technology or simply want to see a machine-generated response to a particular question, this notebook will serve as a comprehensive guide.

## Workflow
1. **Installations**: We'll begin by setting up our environment with the required libraries.
2. **Prerequisites**: Ensure we have access to the Mistral-7B-Instruct-v0.3 model on Hugging Face.
3. **Loading the Model & Tokenizer**: Retrieve the model and tokenizer for our session.
4. **Creating the Llama Pipeline**: Prepare our model for generating responses.
5. **Interacting with Llama**: Prompt the model for answers and explore its capabilities.

Let's dive in!

**First, change runtime to GPU.**


You can play with Mistral-7B-Instruct-v0.3 Chat here:https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3

## Installations

Before we proceed, we need to ensure that the essential libraries are installed:
- `Hugging Face Transformers`: Provides us with a straightforward way to use pre-trained models.
- `PyTorch`: Serves as the backbone for deep learning operations.
- `Accelerate`: Optimizes PyTorch operations, especially on GPU.

In [1]:
!pip install transformers torch accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 58.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

### Prerequisites

To load our desired model, `Mistral-7B-Instruct-v0.3`, we first need to authenticate ourselves on Hugging Face. This ensures we have the correct permissions to fetch the model.

1. Gain access to the model on Hugging Face: [Link] https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3).
2. Use the Hugging Face CLI to login and verify your authentication status.



In [4]:
!huggingface-cli login

# Authenticate with Hugging Face (if required)
# from huggingface_hub import login
# login()  # You'll be prompted to enter your Hugging Face token


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: fineGrained).
The token `irsession_key` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `irse

In [5]:
!huggingface-cli whoami

vmaradhya


### Loading Model & Tokenizer

Here, we are preparing our session by loading both the Llama model and its associated tokenizer.

The tokenizer will help in converting our text prompts into a format that the model can understand and process.

In [10]:
from transformers import AutoTokenizer
import transformers
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.3"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token=True)


/usr/local/lib/python3.11/dist-packages/transformers/models/auto/tokenization_auto.py:823: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


### Creating the Mistral Pipeline

We'll set up a pipeline for text generation.

This pipeline simplifies the process of feeding prompts to our model and receiving generated text as output.

*Note*: This cell takes 2-3 minutes to run

In [11]:
from transformers import pipeline

mistral_pipeline = pipeline(
    "text-generation",  # LLM task
    model=model_name,
    torch_dtype=torch.float16,  # Use float16 for faster inference on GPU
    device_map="auto",  # Automatically map the model to available devices
)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Device set to use cuda:0


### Getting Responses

With everything set up, let's see how Mistral responds to some sample queries.

In [8]:
def get_mistral_response(prompt: str) -> str:
    """
    Generate a response from the Mistral model.

    Parameters:
        prompt (str): The user's input/question for the model.

    Returns:
        str: The model's response.
    """
    sequences = mistral_pipeline(
        prompt,
        do_sample=True,  # Enable sampling for diverse responses
        top_k=10,  # Limit to top-k tokens for diversity
        num_return_sequences=1,  # Return only one sequence
        eos_token_id=tokenizer.eos_token_id,  # End-of-sequence token
        max_new_tokens=256,  # Maximum number of new tokens to generate
    )
    response_text = sequences[0]['generated_text']  # Extract the generated text
    return response_text  # Return the response text

# Make the chatbot conversational
# Initialize conversation history
conversation_history = ""

print("Chatbot: Hello! I'm here to assist you. Type 'bye', 'quit', or 'exit' to end the conversation.")

while True:
    # Get user input
    user_input = input("You: ")
    if user_input.lower() in ["bye", "quit", "exit"]:
        print("Chatbot: Goodbye!")
        break

    # Append new user input to conversation history
    conversation_history += f"[INST] {user_input} [/INST]"

    # Generate response
    full_response = get_mistral_response(conversation_history)

    # Update conversation history with the full response
    conversation_history = full_response

    # Extract only the assistant's latest response
    assistant_response = full_response.split("[/INST]")[-1].strip()

    # Print the assistant's response
    print(f"Chatbot: {assistant_response}")

Chatbot: Hello! I'm here to assist you. Type 'bye', 'quit', or 'exit' to end the conversation.
You: who is sachin tendulkar ?


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Chatbot: Sachin Ramesh Tendulkar is a retired Indian cricketer and a former captain of the Indian national team. He is often referred to as the "God of Cricket" in India due to his prolific batting and significant impact on the sport domestically and internationally. Tendulkar is widely regarded as one of the greatest batsmen in the history of cricket. He started playing cricket at a very young age and made his Test debut in 1989 at the age of 16, becoming the youngest player ever to play Test cricket for India. Over his career, he amassed numerous records including:

1. The first batsman to score a double century in a One Day International (ODI) with 200 not out against South Africa in 2010.
2. The first batsman to score 34 centuries in Tests, a record previously held by Sunil Gavaskar.
3. The first batsman to score 15,000 runs in ODIs.
4. The first player to score 100 international centuries (102 centuries in total across Tests and ODIs).
You: what is his wife name


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Chatbot: Sachin Tendulkar is married to Anjali Mehta. They got married on May 24, 1995, and together they have two children: Sara Tendulkar and Arjun Tendulkar. Anjali is a housewife and a former pediatrician. She comes from a Marwari family and is from Mumbai.
You: bye
Chatbot: Goodbye!


### Problems

After 3-4 prompts, the model stops giving responses. It only outputs the user prompt.

To keep talking to the model, you need to restart the notebook: `Runtime -> Restart Runtime` and run the notebook again...

### More Queries

### Conclusion

Thanks to the Hugging Face Library, creating a pipeline to chat with llama 2 (or any other open-source LLM) is quite easy.

But if you worked a lot with much larger models such as GPT-4, you need to adjust your expectations.